# Neurodesk EDU example: ds003745 trust game from one subject to first-level FEAT

This notebook walks through a simple, novice-friendly workflow in **Neurodesk EDU**:

1. install one OpenNeuro dataset and get one subject  
2. run **fMRIPrep** for that subject and extract FEAT-ready confounds  
3. convert BIDS `events.tsv` files to FSL 3-column EV files  
4. render and run a first-level FEAT model from the Fareri trust-game template  
5. make a first pass at viewing the result in **NiiVue**


**Author**: David V. Smith  

**Date**: March 28, 2026  

**License:** 
<div style="margin-top: 10px;">
    <a href="https://opensource.org/licenses/MIT" target="_blank" style="color: #0066cc;">
        <i class="fas fa-balance-scale"></i> MIT License
    </a>
</div>

**Note:** This notebook was adapted from course materials and the Fareri trust-game workflow code. The notebook keeps the logic visible in notebook cells rather than hiding it inside custom helper functions.


### Citation and Resources

#### Tools included in this workflow
- **fMRIPrep**: Esteban, O., Markiewicz, C. J., Blair, R. W., *et al.* (2019). fMRIPrep: a robust preprocessing pipeline for functional MRI. *Nature Methods, 16*, 111–116.
- **FSL / FEAT**: Jenkinson, M., Beckmann, C. F., Behrens, T. E. J., Woolrich, M. W., & Smith, S. M. (2012). FSL. *NeuroImage, 62*, 782–790.
- **DataLad**: Halchenko, Y. O., Meyer, K., Poldrack, B., *et al.* (2021). DataLad: distributed system for joint management of code, data, and their relationship. *Journal of Open Source Software, 6*, 3262.
- **bidsutils / BIDSto3col.sh**: Tom Nichols and contributors. `bids-standard/bidsutils`.
- **NiiVue / ipyniivue** for interactive image viewing in Jupyter.

#### Workflow this notebook is based on
- `code/fmriprep.sh`
- `code/MakeConfounds.py`
- `code/gen3colfiles.sh`
- `code/L1stats.sh`
- `templates/L1_task-trust_model-01_type-act.fsf`

These files come from the Fareri trust-game workflow repository.

#### Dataset
- **OpenNeuro ds003745** (trust game dataset used in class materials and labs)

#### Educational resources
- Course labs on Neurodesk, fMRIPrep, FEAT, and 3-column event files.


## Table of content
[1. Load software tools and import python libraries](#1.-Load-software-tools-and-import-python-libraries)  
[2. Data preparation](#2.-Data-preparation)  
[3. fMRIPrep for one subject](#3.-Run-fMRIPrep-for-one-subject)  
[4. Make a FEAT confounds file](#4.-Make-a-FEAT-confounds-file)  
[5. Convert events.tsv to 3-column files](#5.-Convert-events.tsv-to-3-column-files)  
[6. Render the FEAT template](#6.-Render-the-FEAT-template)  
[7. Run first-level FEAT](#7.-Run-first-level-FEAT)  
[8. Results](#8.-Results)  
[9. Dependencies in Jupyter/Python](#9.-Dependencies-in-Jupyter/Python)


## 1. Load software tools and import python libraries

In [ ]:
import module
await module.load('fmriprep/20.2.3')
await module.load('fsl/6.0.7.16')
await module.list()


In [ ]:
%%capture
!pip install pandas ipyniivue


In [ ]:
from pathlib import Path
import os
import glob
import pandas as pd
from IPython.display import display, Markdown, Image


## 2. Data preparation

We will keep everything for this example in one folder called `trust_example`.

A few choices here mirror the Fareri workflow:

- the **OpenNeuro dataset** lives in a `bids/` folder
- fMRIPrep writes to a `derivatives/` folder
- FEAT outputs will also go under `derivatives/`
- we copy **one** FEAT template from the Fareri repository
- we install **bidsutils** separately for `BIDSto3col.sh`


In [ ]:
%%bash
set -e

mkdir -p trust_example
cd trust_example

mkdir -p templates bids derivatives scratch

# Copy only the one FEAT template we want from the Fareri workflow repo
curl -L   https://raw.githubusercontent.com/tubric/fareri-2022-neuroimage/main/templates/L1_task-trust_model-01_type-act.fsf   -o templates/L1_task-trust_model-01_type-act.fsf

# Install bidsutils separately so we can use Tom Nichols' BIDSto3col.sh script
if [ ! -d bidsutils ]; then
  git clone --depth 1 https://github.com/bids-standard/bidsutils.git
fi

# Install the OpenNeuro dataset skeleton and get one subject
cd bids
if [ ! -d ds003745 ]; then
  datalad install https://github.com/OpenNeuroDatasets/ds003745.git
fi

cd ds003745
datalad get sub-104/anat
datalad get sub-104/func


In [ ]:
!tree -L 3 trust_example/templates
!tree -L 3 trust_example/bids/ds003745/sub-104


The example below uses:

- **subject**: `104`
- **task**: `trust`
- **run**: `01`

That keeps the notebook manageable, but the same logic can be repeated for the other runs.


## 3. Run fMRIPrep for one subject

The next cell is adapted directly from the logic in `code/fmriprep.sh`.  
The main changes are that the paths are written explicitly for the notebook and we request **MNI152NLin2009cAsym** output because the FEAT template expects the preprocessed BOLD file in that space.

Before running this cell, make sure your FreeSurfer license file exists at:

`~/.license`


In [ ]:
!ls -l ~/.license


In [ ]:
%%bash
set -e

cd trust_example

sub=104
BIDS_DIR=$PWD/bids/ds003745
OUT_DIR=$PWD/derivatives
WORK_DIR=$PWD/scratch
FS_LIC=$HOME/.license

mkdir -p "$OUT_DIR" "$WORK_DIR"

export OMP_NUM_THREADS=1
export ITK_GLOBAL_DEFAULT_NUMBER_OF_THREADS=1

fmriprep "$BIDS_DIR" "$OUT_DIR" participant   --participant-label "$sub"   --stop-on-first-crash   --fs-license-file "$FS_LIC"   --fs-no-reconall   --output-spaces MNI152NLin2009cAsym   --nthreads 12 --omp-nthreads 1 --mem-mb 30000   -w "$WORK_DIR"


When fMRIPrep finishes, the files we care about most for FEAT are:

- the HTML report
- the preprocessed BOLD file for the run we want
- the confounds table for that run

If the HTML report does not open correctly inside Neurodesktop, open it through **JupyterLab** instead.


In [ ]:
!find trust_example/derivatives -name "sub-104.html"
!find trust_example/derivatives -name "*run-01*desc-preproc_bold.nii.gz"
!find trust_example/derivatives -name "*run-01*desc-confounds_timeseries.tsv"


## 4. Make a FEAT confounds file

This cell is adapted from `code/MakeConfounds.py`, but it stays inline and only works on the one confounds file we need for this notebook.

We keep:
- cosine regressors
- non-steady-state regressors
- 6 motion parameters
- the first 6 aCompCor components
- framewise displacement, if it exists

FEAT wants a plain text file **without a header**, so we write the output that way.


In [ ]:
confounds_tsv = Path("trust_example/derivatives/fmriprep/sub-104/func/sub-104_task-trust_run-01_desc-confounds_timeseries.tsv")
confounds = pd.read_csv(confounds_tsv, sep="\t")

cosine_cols = [c for c in confounds.columns if c.startswith("cosine")]
nss_cols = [c for c in confounds.columns if c.startswith("non_steady_state")]
motion_cols = ["trans_x", "trans_y", "trans_z", "rot_x", "rot_y", "rot_z"]
acompcor_cols = [c for c in confounds.columns if c.startswith("a_comp_cor_")][:6]
fd_cols = ["framewise_displacement"] if "framewise_displacement" in confounds.columns else []

keep_cols = cosine_cols + nss_cols + motion_cols + acompcor_cols + fd_cols
feat_confounds = confounds[keep_cols].fillna(0)

out_dir = Path("trust_example/derivatives/fsl/confounds/sub-104")
out_dir.mkdir(parents=True, exist_ok=True)

out_file = out_dir / "sub-104_task-trust_run-01_desc-fslConfounds.tsv"
feat_confounds.to_csv(out_file, sep="\t", header=False, index=False)

print(out_file)
print(feat_confounds.shape)
feat_confounds.head()


## 5. Convert events.tsv to 3-column files

This step follows the same path logic as `code/gen3colfiles.sh`, but we keep it simple and do **one run**.

`BIDSto3col.sh` reads the BIDS events file and writes one 3-column text file per event type.  
Each output file has three columns:

1. onset  
2. duration  
3. weight


In [ ]:
%%bash
set -e

cd trust_example

sub=104
run=01
events_tsv=$PWD/bids/ds003745/sub-${sub}/func/sub-${sub}_task-trust_run-${run}_events.tsv
out_base=$PWD/derivatives/fsl/EVfiles/sub-${sub}/trust/run-${run}

mkdir -p $(dirname "$out_base")

bash bidsutils/BIDSto3col/BIDSto3col/BIDSto3col.sh "$events_tsv" "$out_base"


In [ ]:
!ls trust_example/derivatives/fsl/EVfiles/sub-104/trust
!head -n 5 trust_example/derivatives/fsl/EVfiles/sub-104/trust/run-01*.txt


## 6. Render the FEAT template

This is the part students often find mysterious, so it is worth making explicit.

The Fareri template file already contains the FEAT model structure.  
What changes from subject to subject and run to run are the **paths** and a few settings.

The notebook below follows the same logic as `code/L1stats.sh` for the activation model.

### Placeholders we replace
- `OUTPUT` → where the `.feat` directory should be written  
- `DATA` → the preprocessed BOLD file from fMRIPrep  
- `EVDIR` → the prefix used by the EV timing files  
- `MISSED_TRIAL` → path to the optional missed-trial EV file  
- `EV_SHAPE` → `3` if the missed-trial file exists, otherwise `10` for an empty EV  
- `SMOOTH` → smoothing kernel in mm  
- `CONFOUNDEVS` → the confounds file we just created


In [ ]:
template_text = Path("trust_example/templates/L1_task-trust_model-01_type-act.fsf").read_text()

for token in ["OUTPUT", "DATA", "EVDIR", "MISSED_TRIAL", "EV_SHAPE", "SMOOTH", "CONFOUNDEVS"]:
    print(f"--- lines containing {token} ---")
    for line in template_text.splitlines():
        if token in line:
            print(line)
    print()


The next cell renders a new `.fsf` file. It is adapted from the `sed` command in `code/L1stats.sh`, but each variable is written out here so you can see what is being inserted.


In [ ]:
%%bash
set -e

cd trust_example

TASK=trust
SMOOTH=6
sub=104
run=01

OUTPUT=$PWD/derivatives/fsl/sub-${sub}/L1_task-${TASK}_model-01_type-act_run-${run}_sm-${SMOOTH}
DATA=$PWD/derivatives/fmriprep/sub-${sub}/func/sub-${sub}_task-${TASK}_run-${run}_space-MNI152NLin2009cAsym_desc-preproc_bold.nii.gz
CONFOUNDEVS=$PWD/derivatives/fsl/confounds/sub-${sub}/sub-${sub}_task-${TASK}_run-${run}_desc-fslConfounds.tsv
EVDIR=$PWD/derivatives/fsl/EVfiles/sub-${sub}/${TASK}/run-${run}
MISSED_TRIAL=${EVDIR}_missed_trial.txt

if [ -e "$MISSED_TRIAL" ]; then
  EV_SHAPE=3
else
  EV_SHAPE=10
fi

mkdir -p $(dirname "$OUTPUT")

sed   -e "s@OUTPUT@${OUTPUT}@g"   -e "s@DATA@${DATA}@g"   -e "s@EVDIR@${EVDIR}@g"   -e "s@MISSED_TRIAL@${MISSED_TRIAL}@g"   -e "s@EV_SHAPE@${EV_SHAPE}@g"   -e "s@SMOOTH@${SMOOTH}@g"   -e "s@CONFOUNDEVS@${CONFOUNDEVS}@g"   templates/L1_task-trust_model-01_type-act.fsf   > derivatives/fsl/sub-${sub}/L1_sub-${sub}_task-${TASK}_model-01_run-${run}_act.fsf


In [ ]:
rendered_fsf = Path("trust_example/derivatives/fsl/sub-104/L1_sub-104_task-trust_model-01_run-01_act.fsf")
print(rendered_fsf)
print()
print(rendered_fsf.read_text()[:4000])


## 7. Run first-level FEAT

This final command is the last step from the activation branch of `code/L1stats.sh`.  
It tells FEAT to run the rendered design file.


In [ ]:
%%bash
set -e

cd trust_example
feat derivatives/fsl/sub-104/L1_sub-104_task-trust_model-01_run-01_act.fsf


## 8. Results

The FEAT output directory should now contain the standard report, design matrix, and statistical maps.

A few files worth checking first are:

- `report.html`
- `design.png`
- `stats/thresh_zstat1.nii.gz`
- `stats/thresh_zstat2.nii.gz`
- `stats/thresh_zstat3.nii.gz`


In [ ]:
feat_dir = Path("trust_example/derivatives/fsl/sub-104/L1_task-trust_model-01_type-act_run-01_sm-6.feat")

for rel in [
    "report.html",
    "design.png",
    "stats/thresh_zstat1.nii.gz",
    "stats/thresh_zstat2.nii.gz",
    "stats/thresh_zstat3.nii.gz",
]:
    p = feat_dir / rel
    print(f"{p}: {p.exists()}")


In [ ]:
design_png = feat_dir / "design.png"
if design_png.exists():
    display(Image(filename=str(design_png)))
else:
    print("Run FEAT first, then come back to this cell.")


### Optional: try a quick NiiVue visualization

This cell attempts to overlay a thresholded FEAT result on the FSL MNI brain.  
Because FEAT writes thresholded Z-stat maps as NIfTI files, this usually works well enough for a quick interactive check.

If you want a different contrast, swap `thresh_zstat1.nii.gz` for another statistical map.


In [ ]:
from ipyniivue import NiiVue

mni_bg = "/opt/fsl-6.0.7.16/data/standard/MNI152_T1_2mm_brain.nii.gz"
zmap = feat_dir / "stats" / "thresh_zstat1.nii.gz"

if zmap.exists():
    nv = NiiVue()
    nv.load_volumes([
        {"path": mni_bg, "name": "MNI", "colormap": "gray"},
        {"path": str(zmap), "name": "thresh_zstat1", "colormap": "red", "opacity": 0.6}
    ])
    nv
else:
    print("Run FEAT first, then come back to this cell.")


## 9. Dependencies in Jupyter/Python
- Using the package [watermark](https://github.com/rasbt/watermark) to document system environment and software versions used in this notebook, alongside the Neurodesktop version extracted from the `JUPYTER_IMAGE` environment variable.


In [ ]:
import os

%load_ext watermark

%watermark
%watermark --iversions

neurodesktop_version = os.environ.get('JUPYTER_IMAGE', 'unknown').split(':')[-1]
print(f"Neurodesktop version: {neurodesktop_version}")
